## 1. Tokenization

In [1]:
import tiktoken
tokenizer = tiktoken.get_encoding('gpt2')

In [2]:
text = "I am doing excellent"
enc_ids = tokenizer.encode(text)
print(f"Encoded : {enc_ids}")

dec_text = tokenizer.decode(enc_ids)
print(f"Decoded : {dec_text}")

Encoded : [40, 716, 1804, 6275]
Decoded : I am doing excellent


### 1.1 Tokens of mispelled words

In [3]:
# Unknown words are handled better through BPE : Splitting into known subwords
text = "I am doing excaellent"
enc_ids = tokenizer.encode(text)
dec_text = tokenizer.decode(enc_ids)

print(f"Encoded : {enc_ids}")
print(f"Decoded : {dec_text}")

print("Decoded Step by step : ")
for id in enc_ids:
    sub_word = tokenizer.decode([id])
    print(sub_word)

Encoded : [40, 716, 1804, 2859, 64, 695, 298]
Decoded : I am doing excaellent
Decoded Step by step : 
I
 am
 doing
 exc
a
ell
ent


## 2. Data Loaders & Sequencing

1. Same-length input and target (ex : input size = 1, max_length = output size)
2. Shifted by one token
3. All positions predicted in parallel
4. One next-token loss per position.

In [4]:
# Dataloaders & sequencing

text = "I am going to office"
max_length = 4

# Conceptually:
input_tokens  = ["I",  "am",    "going", "to"]
target_tokens = ["am", "going", "to",    "office"]

# Each position performs next-token prediction:
# "I"              -> "am"
# "I am"           -> "going"
# "I am going"     -> "to"
# "I am going to"  -> "office"

# Loss = CE(pred_1, "am") + CE(pred_2, "going") + CE(pred_3, "to") + CE(pred_4, "office")

# NOT just: input = ["I", "am", "going", "to"] -> target = ["office"]

# Causal masking prevents each position from seeing future tokens. predicting going, for example, the model can see [I, am]

token_ids = tokenizer.encode(text)

i = 0
input_chunk = token_ids[i : i + max_length]
target_chunk = token_ids[i + 1 : i + max_length + 1]

print("Token IDs:", token_ids)
print("Input IDs:", input_chunk)
print("Target IDs:", target_chunk)

print("Input:", tokenizer.decode(input_chunk))
print("Target:", tokenizer.decode(target_chunk))

Token IDs: [40, 716, 1016, 284, 2607]
Input IDs: [40, 716, 1016, 284]
Target IDs: [716, 1016, 284, 2607]
Input: I am going to
Target:  am going to office


## 3. Embedding

In [5]:
import torch.nn as nn
import torch

input_tensor = torch.tensor([2, 1, 0, 3])
context_length = len(input_tensor)
emb_dim=5

In [6]:
# Input Embedding
vocab_size = 1000
token_embedding_layer = nn.Embedding(vocab_size, emb_dim)
token_embedding = token_embedding_layer(input_tensor)
token_embedding.shape

torch.Size([4, 5])

In [7]:
# Positional Embedding
pos_embedding_layer = nn.Embedding(max_length, emb_dim)
pos_embedding = pos_embedding_layer(input_tensor)
pos_embedding.shape

torch.Size([4, 5])

In [8]:
input_embedding = token_embedding + pos_embedding
input_embedding.shape

torch.Size([4, 5])

In [9]:
input_embedding

tensor([[ 1.0938,  0.8718, -1.1066, -0.2855, -1.7346],
        [-1.9790, -1.8719,  2.3355,  3.2052,  1.7472],
        [-1.7953,  1.6514,  0.3713, -0.4712,  1.3895],
        [ 0.0682, -1.3528, -0.2196,  1.3025,  0.8015]], grad_fn=<AddBackward0>)

## Attention

In [10]:
Wk = nn.Linear(emb_dim, emb_dim) # 5, 5
Wq = nn.Linear(emb_dim, emb_dim)
Wv = nn.Linear(emb_dim, emb_dim)

k = Wk(input_embedding) # 4, 5
q = Wq(input_embedding)
v = Wv(input_embedding)

attn_score = q @ k.T  # 4, 4
attn_weight = torch.softmax(attn_score, dim=-1)
context_vector = attn_weight@v  # 4, 5
attn_weight

tensor([[0.1577, 0.3212, 0.1622, 0.3590],
        [0.2184, 0.2113, 0.5051, 0.0652],
        [0.8902, 0.0059, 0.0543, 0.0495],
        [0.1489, 0.2915, 0.4320, 0.1276]], grad_fn=<SoftmaxBackward0>)

In [11]:
mask = torch.triu(torch.ones(context_length, context_length), diagonal=1)
masked = attn_weight.masked_fill(mask.bool(), -torch.inf)
masked_norm = torch.softmax(masked, dim=-1)
masked_norm

tensor([[1.0000, 0.0000, 0.0000, 0.0000],
        [0.5018, 0.4982, 0.0000, 0.0000],
        [0.5416, 0.2237, 0.2348, 0.0000],
        [0.2242, 0.2586, 0.2976, 0.2195]], grad_fn=<SoftmaxBackward0>)

In [12]:
context_vector = masked_norm@v  # 4, 5
context_vector

tensor([[-0.0676,  0.8165, -0.3345, -0.4942, -0.1867],
        [-0.0208,  0.1919, -0.6629, -0.3029,  0.6574],
        [ 0.1473,  0.0313, -0.3093, -0.4344, -0.1039],
        [ 0.1548, -0.2183, -0.3213, -0.2089, -0.0016]], grad_fn=<MmBackward0>)

In [13]:
batch_inputs = torch.stack((input_embedding, input_embedding)) # 2, 4, 5
print(batch_inputs.shape)

Wk = nn.Linear(emb_dim, emb_dim)   # 5, 5
Wq = nn.Linear(emb_dim, emb_dim)
Wv = nn.Linear(emb_dim, emb_dim)

k = Wk(batch_inputs) # 2, 4, 5
q = Wq(batch_inputs)
v = Wv(batch_inputs)

print(q.shape, k.shape)
attn_scores = q @ k.transpose(-2, -1) # 2, 4, 4
attn_weights = torch.softmax(attn_scores, dim=-1)
print(attn_weights.shape)

mask = torch.triu(torch.ones(context_length, context_length), diagonal=1)
masked = attn_weights.masked_fill(mask.bool(), -torch.inf)
attn_masked_norm = torch.softmax(masked, dim=-1)
print(attn_masked_norm)
context_vector = attn_masked_norm @ v
print(context_vector.shape) # 2, 4, 5

torch.Size([2, 4, 5])
torch.Size([2, 4, 5]) torch.Size([2, 4, 5])
torch.Size([2, 4, 4])
tensor([[[1.0000, 0.0000, 0.0000, 0.0000],
         [0.5224, 0.4776, 0.0000, 0.0000],
         [0.3634, 0.3117, 0.3249, 0.0000],
         [0.2005, 0.3191, 0.2689, 0.2115]],

        [[1.0000, 0.0000, 0.0000, 0.0000],
         [0.5224, 0.4776, 0.0000, 0.0000],
         [0.3634, 0.3117, 0.3249, 0.0000],
         [0.2005, 0.3191, 0.2689, 0.2115]]], grad_fn=<SoftmaxBackward0>)
torch.Size([2, 4, 5])


### Multi Head Attention

In [14]:
emb_dim = 10 # this is to split heads equally : emb_dim // num_heads
input_embedding_layer = nn.Embedding(vocab_size, emb_dim)
input_embedding = input_embedding_layer(input_tensor)
print(input_embedding.shape)
## now 10 is divisble by 2 heads

torch.Size([4, 10])


In [15]:
batch_inputs = torch.stack((input_embedding, input_embedding)) # 2, 4, 10
print(batch_inputs.shape)

num_heads = 2
head_dim = emb_dim // 2
Wk = nn.Linear(emb_dim, emb_dim) # 10, 10
Wq = nn.Linear(emb_dim, emb_dim)
Wv = nn.Linear(emb_dim, emb_dim)

k = Wk(batch_inputs) # 2, 4, 10
q = Wq(batch_inputs)
v = Wv(batch_inputs)

k_split = k.view(2, max_length, num_heads, head_dim) # 2, 4, 2, 5
q_split = q.view(2, max_length, num_heads, head_dim)
v_split = v.view(2, max_length, num_heads, head_dim)

k_final = k_split.transpose(1, 2) # 2, 2, 4, 5
q_final = q_split.transpose(1, 2)
v_final = v_split.transpose(1, 2)

attn_score = q_final @ k_final.transpose(-2, -1) # 2, 2, 4, 4

mask = torch.triu(torch.ones(context_length, context_length), diagonal=1)
masked = attn_score.masked_fill(mask.bool(), -torch.inf)
attn_weights_masked = torch.softmax(masked, dim=-1)

context_vector = attn_weights_masked @ v_final # 2, 2, 4, 5
context_vector = context_vector.contiguous().view(2, max_length, emb_dim) # 2, 4, 10

Wo = nn.Linear(emb_dim, emb_dim) # 10, 10
output = Wo(context_vector) # 2, 4, 10
output.shape

torch.Size([2, 4, 10])


torch.Size([2, 4, 10])

## GPT Dummy

In [32]:
# inputs
input_tokens = torch.tensor([4, 2, 1, 0])
target_tokens = torch.tensor([2, 1, 0, 3])

# CONSTANTS
vocab_size = 1000
emb_dim = 512
num_heads = 8
head_dim = emb_dim // num_heads
num_layers = 1
context_length = 5 # useful for positions embedding
max_length = len(input_tokens)
ffn_hidden = 100

# embeddings
token_embedding_layer = nn.Embedding(vocab_size, emb_dim) # 1000, 512
token_embedding = token_embedding_layer(input_tokens) # 4, 512

pos_embedding_layer = nn.Embedding(context_length, emb_dim)
positions = torch.arange(len(input_tokens))
pos_embedding = pos_embedding_layer(positions)  # 4, 512

input_embedding = token_embedding + pos_embedding # 4, 512
target_embedding = token_embedding_layer(target_tokens) # 4, 512

# data loader or batch
input_batch = torch.stack((input_embedding, input_embedding))# 2, 4, 512
target_batch = torch.stack((target_embedding, target_embedding)) # 2, 4, 512

# attention
Wk = nn.Linear(emb_dim, emb_dim) # 512, 512
Wq = nn.Linear(emb_dim, emb_dim)
Wv = nn.Linear(emb_dim, emb_dim)

k = Wk(input_batch) # 2, 4, 512
q = Wq(input_batch)
v = Wv(input_batch)

k_exp = k.view(2, 4, num_heads, head_dim) # 2, 4, 8, 64
k_exp = k_exp.view(2, num_heads, 4, head_dim) # 2, 8, 4, 64

q_exp = q.view(2, 4, num_heads, head_dim) 
q_exp = q_exp.view(2, num_heads, 4, head_dim) 

v_exp = v.view(2, 4, num_heads, head_dim) 
v_exp = v_exp.view(2, num_heads, 4, head_dim) 

attn_score = q_exp @ k_exp.transpose(-2, -1) # 2, 8, 4, 4
mask = torch.triu(torch.ones(max_length, max_length), diagonal=1)
attn_score_masked = attn_score.masked_fill(mask.bool(), -torch.inf)
attn_score_masked = torch.softmax(attn_score_masked, dim=-1) # 2, 8, 4, 4

context_vector = attn_score_masked @ v_exp # 2, 8, 4, 64
context_vector = context_vector.contiguous().view(2, max_length, emb_dim) # 2, 4, 512

Wo = nn.Linear(emb_dim, emb_dim)
context_vector_proj = Wo(context_vector) # 2, 4, 512

# layer norm block
layer_norm = nn.LayerNorm(emb_dim)
context_vector_norm = layer_norm(context_vector_proj) # 2, 4, 512

# FFN
Wffn_1 = nn.Linear(emb_dim, ffn_hidden)
Wffn_2 = nn.Linear(ffn_hidden, emb_dim)
context_vector_ffn_hidden = Wffn_1(context_vector_norm) # 2, 4, 100
context_vector_ffn = Wffn_2(context_vector_ffn_hidden) # 2, 4, 512

# layer norm block
context_vector_norm = layer_norm(context_vector_ffn) # 2, 4, 512

# classify
Wclassify = nn.Linear(emb_dim, vocab_size)
classify_output = Wclassify(context_vector_norm) # 2, 4, 1000
probabilities = torch.softmax(classify_output, dim=-1) # 2, 4, 1000
predicted_tokens = torch.argmax(probabilities, dim=-1) # 2, 4

# loss
import torch.nn.functional as F
logits = classify_output[0]  # [4, 1000]
ce_loss = F.cross_entropy(logits, target_tokens)  # [4, 1000] vs [4]
print("Loss:", ce_loss.item())

Loss: 7.5287981033325195
